# Kaggle runner — exp_0013 / exp_0014 (FT-Transformer reproduction)

Trains the two step-8 reproduction experiments on GPU, on the **frozen 5-fold
partition** shipped as the `s6e7-frozen-folds` dataset (never rebuilt — rule 6):

- **exp_0013** — FT-Transformer (Kawamata recipe via `masamlp`), 13 raw features.
  One variable vs exp_0001: the model family.
- **exp_0014** — same + 39 per-value target-encoding features (`catstat`, fitted
  inside each fold). One variable vs exp_0013: the representation.

Ledger rows record **raw argmax** CV, like every model row; the prior-corrected
decision rule is measured at home as exp_0015/exp_0016 (`cv.run_rule`, cross-fitted).
Libraries pinned to the source notebook's versions. All logic imports from `src/` —
this notebook only orchestrates and displays.

**Carry back** (`kaggle kernels output`): `artifacts/exp_001{3,4}{,_test}.npy` →
`oof/`, and the two new rows of `artifacts/experiments.csv` → the local ledger.

In [ ]:
%pip install -q masamlp==0.3.0 catstat==0.4.0 polars

In [ ]:
import shutil
import subprocess
from pathlib import Path

clone = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/epsilonlog/comp-playground-series-s6e7.git", "repo"],
    capture_output=True, text=True,
)
assert Path("repo/src").exists(), f"clone failed: {clone.stderr}"

inputs = Path("/kaggle/input")
mounted = sorted(p.name for p in inputs.iterdir())
print("mounted inputs:", mounted)

raw = Path("repo/data/raw")
processed = Path("repo/data/processed")
raw.mkdir(parents=True, exist_ok=True)
processed.mkdir(parents=True, exist_ok=True)

comp = inputs / "playground-series-s6e7"
assert comp.exists(), f"competition data not mounted; inputs = {mounted}"
for f in sorted(comp.glob("*.csv")):
    shutil.copy(f, raw / f.name)

folds_src = inputs / "s6e7-frozen-folds" / "folds.parquet"
assert folds_src.exists(), f"frozen-folds dataset not mounted; inputs = {mounted}"
shutil.copy(folds_src, processed / "folds.parquet")

print("raw:", sorted(p.name for p in raw.iterdir()))
print("processed:", sorted(p.name for p in processed.iterdir()))

In [ ]:
import sys

sys.path.insert(0, "repo/src")

import numpy as np
import torch

from s6e7 import cv, decision, features, folds, io
from s6e7.cv import ExperimentConfig

print("cuda:", torch.cuda.is_available())
train, test = io.load_train(), io.load_test()
folds.verify(train)  # the shipped file IS the frozen partition; prove it arrived intact
print("folds verified:", folds.FOLDS_PATH)

Smoke first: does each wrapper run end to end on GPU? 34k rows, 2 epochs, never
logged. The scores are meaningless **by design** — per-value TE cannot be screened at
small n (the source notebook measured it at −0.0017 on a 70k screen vs +0.0012 at
full scale). This cell only proves the plumbing before an hour of training.

In [ ]:
smoke = train.head(34_000)
for name in ("ftt", "ftt_te"):
    r = cv.run(
        ExperimentConfig(exp_id=f"smoke_{name}", model=name, params={"n_epochs": 2}),
        train=smoke,
        log=False,
    )
    print(f"{name}: plumbing ok, {r.runtime_s:.0f}s (scores meaningless at this n)")

In [ ]:
result_13 = cv.run(
    ExperimentConfig(
        exp_id="exp_0013",
        model="ftt",
        parent="exp_0001",
        changed="new family: FT-Transformer (masamlp, Kawamata recipe), raw features",
    ),
    train=train,
    test=test,
    if_logged="skip",
)
print(f"exp_0013  cv_mean={result_13.cv_mean:.5f}  cv_std={result_13.cv_std:.5f}  (raw argmax)")
print("fold scores:", [round(s, 5) for s in result_13.fold_scores])

In [ ]:
result_14 = cv.run(
    ExperimentConfig(
        exp_id="exp_0014",
        model="ftt_te",
        parent="exp_0013",
        changed="add 39 per-value target-encoding features (catstat, fitted inside the fold)",
    ),
    train=train,
    test=test,
    if_logged="skip",
)
print(f"exp_0014  cv_mean={result_14.cv_mean:.5f}  cv_std={result_14.cv_std:.5f}  (raw argmax)")
print("fold scores:", [round(s, 5) for s in result_14.fold_scores])

Preview of the number that actually matters — the prior-corrected score. In-sample
on the full OOF (fine as a preview: two parameters on 550k rows overfit by ≈0 — see
exp_0009's cache); the honest cross-fitted rows are logged at home as exp_0015/0016.

In [ ]:
y = features.encode_target(train[io.TARGET])
for exp_id in ("exp_0013", "exp_0014"):
    proba = np.load(f"repo/oof/{exp_id}.npy")
    multipliers, score = decision.search(proba, y)
    print(f"{exp_id}: rule preview {score:.5f}  multipliers {np.round(multipliers, 3).tolist()}")

In [ ]:
!mkdir -p /kaggle/working/artifacts
!cp repo/oof/exp_0013.npy repo/oof/exp_0013_test.npy repo/oof/exp_0014.npy repo/oof/exp_0014_test.npy repo/experiments.csv /kaggle/working/artifacts/
!rm -rf repo
!ls -l /kaggle/working/artifacts